# Cell-Painting MoA Classification — Channel-Dropout Multiplex

**Task.** Predict the mechanism-of-action (MoA, 12 classes) of a compound from a 3-panel
Cell-Painting image where **one panel is blacked out**. Submission is a probability per class;
the metric rewards calibrated log-loss, so *confidently wrong* predictions are punished hard.

**Two structural difficulties drive every design choice below:**
1. **Compound confounding.** Three classes (`moa_02/05/09`) are backed by a *single compound*
   each. Their label is collinear with that compound's batch/staining signature, so a CNN can
   "recognize the compound" instead of the biology — and that shortcut does not transfer to the
   compound-blind test set. We therefore (a) validate with **compound-grouped** CV so our score
   reflects *unseen-compound* generalization, and (b) **damp confidence** on the single-compound
   classes at calibration.
2. **One missing panel.** Exactly one of the three 160×160 panels is `(0,0,0)` in *both* train
   and test (`masked_region` names which). Train and test already share this distribution, so
   there is **no channel-dropout augmentation to add** — a second mask would be off-distribution.
   We feed `masked_region` as a one-hot so the head knows which panel is absent.

**Approach.** Frozen ImageNet `convnext_small` features (chosen over tiny/base by compound-blind
OOF) + a light MLP head, K-fold ensemble, then temperature + sign-correct thin-class shrinkage +
ensemble-disagreement (BALD) blending toward the uniform prior. Runs self-contained in a few
minutes on the A10G. No LLM is used anywhere in the pipeline.


In [ ]:
# ── Imports & config — Kaggle-Docker libraries only ──────────────────────────────
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
import torchvision
from torchvision.models import convnext_small, ConvNeXt_Small_Weights
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize_scalar
warnings.filterwarnings("ignore")

t0 = time.time()
DATA = Path("./dataset/public")          # Eris mounts the competition data here
OUT  = Path("./working"); OUT.mkdir(parents=True, exist_ok=True)  # write submission here
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 12
LABELS    = [f"moa_{i:02d}" for i in range(NUM_CLASSES)]     # sorted == natural order
LAB2IDX   = {l: i for i, l in enumerate(LABELS)}
PROB_COLS = [f"prob_{l}" for l in LABELS]
N_PANELS, PANEL = 3, 160                                     # 480 = 3 x 160
REGION2PANEL = {"region_0": 0, "region_1": 1, "region_2": 2}
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
def panel_slice(p): return slice(p * PANEL, (p + 1) * PANEL)
print("device:", DEVICE)


## 1. Load data and verify the load-bearing invariants
We **re-derive** which classes are single-compound from the data rather than trusting a constant —
a silent index desync here would damage the exact classes we mean to protect.


In [ ]:
train_df = pd.read_csv(DATA / "train.csv")
test_df  = pd.read_csv(DATA / "test.csv")
samp_df  = pd.read_csv(DATA / "sample_submission.csv")
print(f"train={len(train_df)}  test={len(test_df)}  classes={train_df.moa_label.nunique()}  "
      f"compounds={train_df.compound_id.nunique()}")

# integer encoding moa_0X -> X, verified against the data
assert LABELS == sorted(train_df.moa_label.unique().tolist())
Ytr_all = train_df.moa_label.map(LAB2IDX).values

# single-compound ("thin") classes are confounded; 2-compound classes are the LOCO analog
comp_per_class = train_df.groupby("moa_label").compound_id.nunique()
THIN = sorted(LAB2IDX[l] for l, n in comp_per_class.items() if n == 1)   # confounded
LOCO = sorted(LAB2IDX[l] for l, n in comp_per_class.items() if n == 2)   # unseen-compound analog
print("single-compound (thin) classes:", THIN, "=", [LABELS[i] for i in THIN])
print("2-compound (LOCO) classes:      ", LOCO)
assert THIN == [2, 5, 9], f"unexpected thin classes {THIN}"

# spot-check the exactly-one-black-panel invariant on a sample (full check is O(N) but cheap)
def zero_panels(img): return [p for p in range(N_PANELS) if not img[:, panel_slice(p), :].any()]
for _, r in train_df.sample(12, random_state=0).iterrows():
    im = np.asarray(Image.open(DATA / r.panel_path).convert("RGB"))
    assert zero_panels(im) == [REGION2PANEL[r.masked_region]]
print("verified: exactly one black panel per image, matching masked_region")


## 2. Frozen `convnext_small` features (defensive weight load)
At N=288 images, **fine-tuning a big backbone overfits**; we use it *frozen* as a feature
extractor and train only a light head. `convnext_small` beat `tiny` and `base` on compound-blind
OOF (base over-fits the tiny dataset). Weight loading is defensive: use ImageNet weights if
reachable, otherwise proceed (and log it) so the notebook never crashes.


In [ ]:
class Backbone(nn.Module):
    """convnext_small -> 768-d pooled features (drop the final ImageNet Linear)."""
    def __init__(self, net):
        super().__init__()
        self.features, self.avgpool, self.norm = net.features, net.avgpool, net.classifier[0]
    def forward(self, x):
        x = self.norm(self.avgpool(self.features(x)))
        return torch.flatten(x, 1)

def load_backbone():
    tag = "pretrained(IMAGENET1K_V1)"
    try:
        net = convnext_small(weights=ConvNeXt_Small_Weights.IMAGENET1K_V1)
    except Exception as e:
        print("WARN: pretrained weights unavailable (", e, ") -> random init"); tag = "random"
        net = convnext_small(weights=None)
    bb = Backbone(net).eval().to(DEVICE)
    for p in bb.parameters(): p.requires_grad_(False)
    print("backbone:", tag)
    return bb

FEAT_DIM = 768
_backbone = load_backbone()

@torch.no_grad()
def extract(df, bb, batch=16):
    feats, masks = [], []
    for i in range(0, len(df), batch):
        chunk = df.iloc[i:i + batch]
        xs, ms = [], []
        for _, r in chunk.iterrows():
            im = np.asarray(Image.open(DATA / r.panel_path).convert("RGB"), np.float32) / 255.0
            x = (np.transpose(im, (2, 0, 1)) - IMAGENET_MEAN) / IMAGENET_STD  # composite as-is
            xs.append(x)
            oh = np.zeros(N_PANELS, np.float32); oh[REGION2PANEL[r.masked_region]] = 1.0
            ms.append(oh)
        f = bb(torch.tensor(np.stack(xs)).to(DEVICE)).cpu().numpy()
        feats.append(f); masks.append(np.stack(ms))
    return np.concatenate(feats), np.concatenate(masks)

Xtr, Mtr = extract(train_df, _backbone)
Xte, Mte = extract(test_df, _backbone)
print(f"features: train{Xtr.shape} test{Xte.shape}  ({time.time()-t0:.0f}s)")


## 3. Leakage-free compound-grouped CV (thin compounds pinned into every train fold)
We group folds by `compound_id` so a compound never appears in both train and validation — this
makes OOF reflect *unseen-compound* generalization, the test condition. The three single-compound
classes are **pinned into every training fold** (never validated): a fold that held them out
would have zero examples of that class and could not learn it. Consequently thin classes have
**no OOF** — that is correct, not a bug, and is why we calibrate them by transfer (below).


In [ ]:
def make_folds(df, n=5):
    y, g = df.moa_label.map(LAB2IDX).values, df.compound_id.values
    thin_idx  = [i for i in range(len(df)) if y[i] in THIN]
    dense_idx = [i for i in range(len(df)) if y[i] not in THIN]
    sgkf = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=SEED)
    folds = []
    for tr, va in sgkf.split(dense_idx, [y[dense_idx[j]] for j in range(len(dense_idx))],
                             [g[dense_idx[j]] for j in range(len(dense_idx))]):
        train_idx = sorted([dense_idx[j] for j in tr] + thin_idx)   # thin pinned into train
        val_idx   = sorted([dense_idx[j] for j in va])
        # guard: no compound spans train & val
        assert not ({g[i] for i in train_idx} & {g[i] for i in val_idx})
        folds.append((train_idx, val_idx))
    return folds

class Head(nn.Module):
    def __init__(self, d, nmask=N_PANELS, C=NUM_CLASSES, hid=256, pdrop=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d + nmask, hid), nn.BatchNorm1d(hid),
                                 nn.ReLU(True), nn.Dropout(pdrop))
        self.cls = nn.Linear(hid, C)
    def forward(self, feat, mask):
        return self.cls(self.net(torch.cat([feat, mask], 1)))

def train_head(Xt, Mt, Yt, Xv, Mv, epochs=120, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    head = Head(Xt.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    ce = nn.CrossEntropyLoss(label_smoothing=0.1)   # smoothing regularizes; not a shortcut fix
    Xt_ = torch.tensor(Xt, device=DEVICE); Mt_ = torch.tensor(Mt, device=DEVICE)
    Yt_ = torch.tensor(Yt, device=DEVICE); n = len(Xt_); bs = min(64, n)
    head.train()
    for _ in range(epochs):
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            if idx.numel() < 2: continue
            opt.zero_grad(); ce(head(Xt_[idx], Mt_[idx]), Yt_[idx]).backward(); opt.step()
        sch.step()
    head.eval()
    with torch.no_grad():
        vl = head(torch.tensor(Xv, device=DEVICE), torch.tensor(Mv, device=DEVICE)).cpu().numpy()
    return head, vl

folds = make_folds(train_df)
oof = np.full((len(train_df), NUM_CLASSES), np.nan, np.float32)
heads = []
for k, (tr, va) in enumerate(folds):
    head, vl = train_head(Xtr[tr], Mtr[tr], Ytr_all[tr], Xtr[va], Mtr[va], seed=SEED + k)
    oof[va] = vl; heads.append(head)

def softmax_np(z, T=1.0):
    z = z / T; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def logloss_np(p, y): p = np.clip(p, 1e-9, 1); return float(-np.log(p[np.arange(len(y)), y]).mean())

dmask = ~np.isnan(oof[:, 0]); yd = Ytr_all[dmask]; od = oof[dmask]
n_dense = int((~np.isin(Ytr_all, THIN)).sum()); n_thin = int(np.isin(Ytr_all, THIN).sum())
print(f"OOF coverage: dense={int(dmask.sum())}/{n_dense}, thin={n_thin} (no OOF by design)")
print(f"[dense OOF] logloss={logloss_np(softmax_np(od), yd):.3f} (uniform {np.log(12):.3f})  "
      f"macroF1={f1_score(yd, od.argmax(1), average='macro', labels=list(range(NUM_CLASSES)), zero_division=0):.3f}")


## 4. Calibration — protect log-loss on the shifted / thin classes
- **Temperature** `T` is fit on the dense-class OOF (legitimate — those classes have held-out-
  compound predictions).
- **Thin-class shrinkage.** We can't estimate confidence for the single-compound classes from
  their own data, so we shrink their probability mass by a factor fit on the **2-compound (LOCO)
  classes** — the closest measurable analog to "a class seen through one (other) compound" — and
  transfer it as a lower bound. Shrinkage is applied in probability space and **renormalized**
  (sign-independent), unlike dividing logits which would *raise* thin-class mass on non-thin rows.
- **Ensemble disagreement (BALD)** `H(mean) − mean_k H(p_k)` blends uncertain predictions mildly
  toward the uniform prior at inference.


In [ ]:
def shrink(p, cls, s):
    q = p.copy(); q[:, cls] *= s; return q / q.sum(1, keepdims=True)

T = float(np.exp(minimize_scalar(
    lambda lt: logloss_np(softmax_np(od, np.exp(lt)), yd),
    bounds=(np.log(0.25), np.log(10.0)), method="bounded").x))
base = softmax_np(od, T)
thin_shrink = float(minimize_scalar(
    lambda s: logloss_np(shrink(base, LOCO, s), yd),
    bounds=(0.05, 1.0), method="bounded").x)   # LOCO-fit lower bound, transferred to THIN
print(f"T={T:.3f}  thin_shrink={thin_shrink:.3f}")
print(f"[dense OOF] raw={logloss_np(softmax_np(od),yd):.3f}  temp={logloss_np(base,yd):.3f}  "
      f"temp+LOCO-shrink={logloss_np(shrink(base,LOCO,thin_shrink),yd):.3f}")
assert logloss_np(base, yd) <= logloss_np(softmax_np(od), yd) + 1e-6   # temp never hurts


## 5. Inference → `./working/submission.csv`
Ensemble the K fold heads, apply the calibration transform, and write probabilities in
`sample_submission` order with validity guards (144 rows, each sums to 1, non-negative).


In [ ]:
Xt_ = torch.tensor(Xte, device=DEVICE); Mt_ = torch.tensor(Mte, device=DEVICE)
with torch.no_grad():
    stack = np.stack([h(Xt_, Mt_).cpu().numpy() for h in heads])      # (K, Nte, 12)

Pk = softmax_np(stack.reshape(-1, NUM_CLASSES), T).reshape(stack.shape)
mean = Pk.mean(0)
ent = lambda p: -(p * np.log(np.clip(p, 1e-12, 1))).sum(1)
bald = np.clip(ent(mean) - np.stack([ent(Pk[k]) for k in range(len(Pk))]).mean(0), 0, None)
bald = bald / max(bald.max(), 1e-9)
probs = (1 - (bald**2)[:, None]) * shrink(mean, THIN, thin_shrink) + (bald**2)[:, None] * (1.0/NUM_CLASSES)

by_id = {test_df.sample_id.iloc[i]: probs[i] for i in range(len(test_df))}
order = samp_df.sample_id.tolist()
assert set(order) == set(by_id), "test ids != sample_submission ids"
sub = pd.DataFrame([[sid] + by_id[sid].tolist() for sid in order], columns=["sample_id"] + PROB_COLS)
assert np.allclose(sub[PROB_COLS].sum(axis=1), 1.0, atol=1e-5) and (sub[PROB_COLS].values >= 0).all()
sub.to_csv(OUT / "submission.csv", index=False)
print(f"wrote {OUT/'submission.csv'}: {len(sub)} rows, {len(PROB_COLS)} prob cols; "
      f"thin-mass mean={probs[:, THIN].sum(1).mean():.3f}; total elapsed={time.time()-t0:.0f}s")
